<a href="https://colab.research.google.com/github/sridevi-t/project_using_dataset/blob/main/mbti112.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# =========================
# 2. DOWNLOAD STOPWORDS
# =========================
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# =========================
# 3. LOAD DATASET (FIXED)
# =========================
df = pd.read_csv(
    '/content/sample_data/mbti_1.xlsx.csv',
    engine='python',
    on_bad_lines='skip'
)

print("Dataset loaded successfully!")
print(df.shape)
print(df.columns)

# =========================
# 4. CLEAN TEXT FUNCTION
# =========================
def clean_text(text):
    text = str(text)
    text = text.replace("|||", " ")   # important for MBTI dataset
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

# Apply cleaning
df['cleaned_posts'] = df['posts'].apply(clean_text)

# =========================
# 5. ENCODE LABELS
# =========================
le = LabelEncoder()
df['label'] = le.fit_transform(df['type'])

# =========================
# 6. TRAIN-TEST SPLIT (IMPROVED)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_posts'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# =========================
# 7. TF-IDF VECTORIZATION
# =========================
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# =========================
# 8. MODEL TRAINING
# =========================
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# =========================
# 9. EVALUATION
# =========================
y_pred = model.predict(X_test_tfidf)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# =========================
# 10. PREDICTION FUNCTION
# =========================
def predict_mbti(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    pred = model.predict(vector)
    return le.inverse_transform(pred)[0]

# =========================
# 11. USER INPUT TEST
# =========================
while True:
    text = input("\nEnter text (or type 'exit'): ")
    if text.lower() == 'exit':
        break
    print("Predicted MBTI:", predict_mbti(text))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset loaded successfully!
(869, 2)
Index(['type', 'posts'], dtype='object')


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Accuracy: 0.46551724137931033

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00        13
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         2
           8       0.53      0.55      0.54        29
           9       0.40      0.92      0.56        39
          10       0.55      0.48      0.51        23
          11       0.51      0.67      0.58        27
          12       0.00      0.00      0.00         3
          13       0.00      0.00      0.00         4
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00         8

    accuracy            

In [1]:
!pip install flask

In [2]:
%%writefile app.py
# paste your FULL Flask code here
from flask import Flask, render_template, request
import pandas as pd
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

app = Flask(__name__)

# =========================
# LOAD + TRAIN MODEL (once)
# =========================
df = pd.read_csv('/content/sample_data/mbti_1.xlsx.csv', engine='python', on_bad_lines='skip')

def clean_text(text):
    text = str(text)
    text = text.replace("|||", " ")
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['cleaned_posts'] = df['posts'].apply(clean_text)

le = LabelEncoder()
df['label'] = le.fit_transform(df['type'])

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_posts'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# =========================
# PREDICTION FUNCTION
# =========================
def predict_mbti(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    pred = model.predict(vector)
    return le.inverse_transform(pred)[0]

# =========================
# ROUTES
# =========================
@app.route('/', methods=['GET', 'POST'])
def home():
    prediction = ""
    if request.method == 'POST':
        user_text = request.form['text']
        if user_text.strip() != "":
            prediction = predict_mbti(user_text)
        else:
            prediction = "Please enter some text!"
    return render_template('index.html', prediction=prediction)

# =========================
# RUN APP
# =========================
if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

Overwriting app.py


In [82]:
!ls

app.py	drive  sample_data  templates


In [3]:
!mkdir -p templates


In [4]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>MBTI Predictor</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            background: linear-gradient(to right, #667eea, #764ba2);
            text-align: center;
            color: white;
            margin: 0;
            padding: 0;
        }

        .container {
            margin-top: 80px;
        }

        h1 {
            font-size: 40px;
            margin-bottom: 20px;
        }

        textarea {
            width: 60%;
            height: 120px;
            padding: 15px;
            border-radius: 10px;
            border: none;
            font-size: 16px;
            resize: none;
        }

        button {
            margin-top: 15px;
            padding: 12px 25px;
            font-size: 16px;
            background-color: #ff7a18;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
        }

        button:hover {
            background-color: #ff4e00;
        }

        .result {
            margin-top: 30px;
            font-size: 24px;
            background-color: rgba(0,0,0,0.2);
            display: inline-block;
            padding: 15px 25px;
            border-radius: 10px;
        }

        .footer {
            position: fixed;
            bottom: 10px;
            width: 100%;
            font-size: 14px;
            opacity: 0.8;
        }
    </style>
</head>

<body>

    <div class="container">
        <h1>🧠 MBTI Personality Predictor</h1>

        <form method="POST">
            <textarea name="text" placeholder="Type something about yourself..."></textarea><br>
            <button type="submit">Predict</button>
        </form>

        {% if prediction %}
            <div class="result">
                Predicted MBTI: <b>{{ prediction }}</b>
            </div>
        {% endif %}
    </div>

    <div class="footer">
        Built with using Machine Learning
    </div>

</body>
</html>

Overwriting templates/index.html


In [5]:
!pip install flask pyngrok

In [6]:
from pyngrok import ngrok

ngrok.kill()

In [95]:
!ngrok config add-authtoken 3Chq5wdOuQS5nNyD5z23oJlceHq_aTV1uiPu3mPK36wSQLfn

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [7]:
from pyngrok import ngrok
import subprocess
import time

# Start Flask app
process = subprocess.Popen(["python", "app.py"])

# Wait for server to start
time.sleep(5)

# Start ngrok tunnel
url = ngrok.connect(5000)
print("Open this:", url)

Open this: NgrokTunnel: "https://sensuous-halved-unboxed.ngrok-free.dev" -> "http://localhost:5000"


In [90]:

!python app.py

  File "/content/app.py", line 69
    user_text = request.form['text']
    ^^^^^^^^^
IndentationError: expected an indented block after 'if' statement on line 68
